# Forward inference — score 2025-2027 with the deployed model (E)

Trains the winning **E = crash + infrastructure + spatial** model once on the verified
corrected set (2016-2021 -> 2022-2024), then scores the **forward** candidates below the
City's 5-crash screen with 2016-2024 features to predict **2025-2027** KSI emergence.

This is predict-only: the model never sees the forward labels during fitting, so the
provisional 2025 recall check at the end is leakage-free.

### Upload these 3 files
| file | on your machine, in |
|---|---|
| `verified_model_input.parquet` | `data\\model\\` |
| `forward_model_input.parquet` | `data\\model\\` |
| `frozen_params.json` | `data\\model\\` |


In [ ]:
!pip -q install xgboost 2>/dev/null
import xgboost; print("xgboost", xgboost.__version__, "| ready")


In [ ]:
from google.colab import files
up = files.upload()
need = {"verified_model_input.parquet","forward_model_input.parquet","frozen_params.json"}
print("\nMISSING:", (need - set(up)) or "none - good to go")


In [ ]:
# Train E on verified, score forward (2025-2027)
import json, numpy as np, pandas as pd, xgboost as xgb

COST = 5_175_524; EFF = 0.30; PROG = 3_200_000; CITY_MIN = 5

CRASH = ["crashes_36mo","crashes_72mo","ped_crashes_72mo","bike_crashes_72mo","broadside_72mo",
 "left_turn_72mo","dui_72mo","night_72mo","ped_row_violation_72mo","years_since_last_crash",
 "distinct_crash_days_72mo","worst_severity_72mo","crash_trend_slope","emergence_velocity",
 "emergence_acceleration","mann_kendall_tau","changepoint_prob","ewma_crashes","momentum_ratio","covid_period_share"]
SPAT = ["nbr_crashes_150m","nbr_crashes_400m","nbr_ksi_400m","node_density_400m","n_hotspots_400m","dist_nearest_hotspot_m"]
NON_FEAT = set(CRASH + SPAT + ["intersection_id","KSI_label","spatial_block","persistence_baseline_score","crashes_feat"])

tr = pd.read_parquet("verified_model_input.parquet")
fw = pd.read_parquet("forward_model_input.parquet")
INFRA = [c for c in tr.columns if c not in NON_FEAT]
E = CRASH + INFRA + SPAT                       # the deployed feature set
print(f"E model: {len(E)} features ({len(CRASH)} crash + {len(INFRA)} infra + {len(SPAT)} spatial)")

fp = json.load(open("frozen_params.json"))["frozen"]
params = dict(objective="reg:tweedie",
    tweedie_variance_power=float(fp["tweedie_variance_power"]), max_depth=int(fp["max_depth"]),
    learning_rate=float(fp["learning_rate"]), n_estimators=int(fp["n_estimators"]),
    reg_alpha=float(fp["reg_alpha"]), reg_lambda=float(fp["reg_lambda"]),
    min_child_weight=int(fp["min_child_weight"]), subsample=float(fp["subsample"]),
    colsample_bytree=float(fp["colsample_bytree"]), random_state=42, verbosity=0)

# TRAIN on verified sites below the City screen
tr_c = tr[tr["crashes_feat"] < CITY_MIN]
model = xgb.XGBRegressor(**params)
model.fit(tr_c[E].fillna(0.0).values.astype(float), tr_c["KSI_label"].values.astype(float))
print(f"trained on {len(tr_c)} verified city-ignored sites ({int((tr_c.KSI_label>=1).sum())} KSI positives)")

# SCORE forward candidates below the City screen (predict-only)
fw_c = fw[fw["crashes_feat"] < CITY_MIN].copy()
fw_c["risk_score"] = model.predict(fw_c[E].fillna(0.0).values.astype(float))
fw_c = fw_c.sort_values("risk_score", ascending=False).reset_index(drop=True)
print(f"scored {len(fw_c)} forward city-ignored candidates for 2025-2027\n")

# deployable outputs
cols_out = ["intersection_id","crashes_feat","risk_score","KSI_label"] + [c for c in ("lon","lat") if c in fw_c.columns]
fw_c[cols_out].to_parquet("forward_scored_full.parquet", index=False)      # feeds the dashboard export
fw_c.head(500)[cols_out].to_csv("forward_city_ignored_top500.csv", index=False)  # the live shortlist

# provisional 2025 recall (predict-only, leakage-free; only 1 of 3 label years complete)
y = fw_c["KSI_label"].values
pos = int((y >= 1).sum()); base = pos / len(fw_c)
print(f"provisional 2025 check: {pos} partial-year KSI positives in {len(fw_c)} candidates (~{100*base:.2f}% base rate)")
print("PROVISIONAL -- 2025 only of the 2025-2027 window; treat as directional.\n")
for K in (100, 200, 500, 1000):
    hits = int((y[:K] >= 1).sum()); ev = int(y[:K].sum())
    print(f"  top-{K:4d}: {hits:3d} sites hit, {ev:3d} events, {hits/(K*base):.1f}x random, ${ev*COST*EFF/1e6:.1f}M prevented @30%")

print("\nSaved: forward_city_ignored_top500.csv  (the 2025-2027 live shortlist)")
print("Saved: forward_scored_full.parquet       (all scored candidates -> dashboard export)")
try:
    files.download("forward_city_ignored_top500.csv")
except Exception:
    pass


## What you get

- **`forward_city_ignored_top500.csv`** — the deployable shortlist: the 500 intersections
  the City's 5-crash screen ignores, ranked by predicted 2025-2027 KSI risk. This is the
  list to hand the City and to serve on the map.
- **`forward_scored_full.parquet`** — every scored candidate; this is what feeds
  `build_export_panel` so the dashboard shows the E model instead of the old crash-only run.
- The provisional 2025 recall is a leakage-free sanity check (the model never trained on
  those labels). Only 2025 of the 2025-2027 window is complete, so read it as directional
  until the next SWITRS export lands.

Send me the recall numbers and I'll tell you how the deployed model looks out-of-sample,
then wire `forward_scored_full.parquet` into the dashboard export.
